# Graph Attention Network #

## Connectivity Matrix ##

In [15]:
import numpy as np
import torch
from enigmatoolbox.datasets import load_fc, load_sc

def apply_threshold(matrix, top_pct):
    # Keep only the top % of connections (ignoring zeros/diagonal)
    threshold_val = np.percentile(matrix[matrix > 0], 100 - top_pct)
    return np.where(matrix >= threshold_val, matrix, 0)

def get_edge_index(adj_matrix):
    sources, targets = np.nonzero(adj_matrix)
    edge_weights = adj_matrix[sources, targets]
    
    edge_index = torch.tensor(np.vstack((sources, targets)), dtype=torch.long)
    edge_attr = torch.tensor(edge_weights, dtype=torch.float)
    return edge_index, edge_attr

def load_connectivity_414(conn_type='FC', top_pct=10.0):
    """
    Fetches the HCP connectome using the ENIGMA Toolbox.
    """
    if conn_type == 'FC':
        ctx_fc, _, sub_ctx_fc, _ = load_fc('schaefer_400')
        # subcortico-subcortical connections are currently missing from the enigmatoolbox dataset
        # so we will use an identity matrix or zeros here for now, or just self-connections
        sub_sub_fc = np.eye(14) 
        
        matrix = np.block([
            [ctx_fc,          sub_ctx_fc.T],  # Top-left (400x400), Top-right (400x14)
            [sub_ctx_fc,      sub_sub_fc]     # Bottom-left (14x400), Bottom-right (14x14)
        ])
    elif conn_type == 'SC':
        # structural connectivity
        ctx_sc, _, sub_ctx_sc, _ = load_sc('schaefer_400')
        sub_sub_sc = np.eye(14)
        
        matrix = np.block([
            [ctx_sc,          sub_ctx_sc.T],
            [sub_ctx_sc,      sub_sub_sc]
        ])
    else:
        raise ValueError("conn_type must be 'FC' or 'SC'")
        
    adj = apply_threshold(matrix, top_pct)
    return get_edge_index(adj)

# Test fetching the Functional Connectivity 
edge_index, edge_attr = load_connectivity_414(conn_type='FC', top_pct=10.0)
print(f"Edge Index Shape: {edge_index.shape}")
print(f"Edge Attr Shape: {edge_attr.shape}")

Edge Index Shape: torch.Size([2, 16596])
Edge Attr Shape: torch.Size([16596])


## PyTorch Geometric Dataset
We wrap the processed `.npz` files into a standard PyTorch Geometric InMemoryDataset. We only store the BOLD beta for each of the 414 regions, along with a distinct `node_id` (0 to 413). The static structural identity will be learned dynamically via an `nn.Embedding` layer in the model.

In [16]:
import os
import json
import torch
import numpy as np
from torch_geometric.data import Data, InMemoryDataset
from torch.utils.data import ConcatDataset

class BOLD5000GraphDataset(InMemoryDataset):
    def __init__(self, root, edge_index, edge_attr, subject='CSI1', transform=None):
        self.edge_index = edge_index
        self.edge_attr = edge_attr
        self.subject = subject
        self.data_path = os.path.join(root, subject, f"{subject}_schaefer414.npz")
        
        with open('SemanticLabels.json', 'r') as f:
            self.ontology = json.load(f)
        
        self.category_to_id = {cat: i for i, cat in enumerate(self.ontology.keys())}
        
        self.label_to_category = {}
        for category, details in self.ontology.items():
            for label in details['labels']:
                self.label_to_category[label.lower()] = category
                
        super().__init__(root, transform)
        
        # In PyTorch 2.6+, weights_only=True is the default. We need weights_only=False 
        # to load the complex PyTorch Geometric Data objects correctly.
        self.data, self.slices = torch.load(self.processed_paths[0], weights_only=False)

    @property
    def processed_file_names(self):
        return [f'{self.subject}_graph_data_v3.pt']

    def process(self):
        print(f"Processing {self.subject} BOLD fMRI into brain graphs...")
        archive = np.load(self.data_path, allow_pickle=True)
        betas = archive['betas']
        labels = archive['labels']
        
        # Extract metadata fields needed for downstream visual mapping
        imgnames = archive['imgnames']
        sessions = archive['sessions']
        local_idxs = archive['local_idxs']
        
        data_list = []
        skipped = 0
        
        # Create a static vector of node IDs [0, 1, 2, ..., 413]
        num_nodes = betas.shape[1]
        node_ids = torch.arange(num_nodes, dtype=torch.long)
        
        for i in range(len(betas)):
            trial_labels = [l.lower() for l in labels[i]]
            found_categories = set(
                self.label_to_category[l] for l in trial_labels if l in self.label_to_category
            )
            
            # Skip trials with conflicting or missing labels for clean training
            if len(found_categories) != 1:
                skipped += 1
                continue
                
            target_category = list(found_categories)[0]
            y = torch.tensor([self.category_to_id[target_category]], dtype=torch.long)
            
            # Extract trial beta [414, 1]
            trial_beta = torch.tensor(betas[i], dtype=torch.float).unsqueeze(1)
            
            # Inject metadata directly into the Data object for tracking
            graph_data = Data(
                x=trial_beta,           # Only the beta signal [414, 1]
                node_id=node_ids,       # The structural identity [414]
                edge_index=self.edge_index, 
                edge_attr=self.edge_attr, 
                y=y,
                imgname=str(imgnames[i]),
                session=int(sessions[i]),
                local_idx=int(local_idxs[i]),
                subject=self.subject
            )
            data_list.append(graph_data)
            
        print(f"[{self.subject}] Created {len(data_list)} graphs. Skipped {skipped} multi-label/unknown trials.")
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])

# Instantiate datasets for all 4 subjects and concatenate them into one massive dataset
subjects = ['CSI1', 'CSI2', 'CSI3', 'CSI4']
all_datasets = []

for subj in subjects:
    ds = BOLD5000GraphDataset(
        root='processed_data', 
        edge_index=edge_index, 
        edge_attr=edge_attr, 
        subject=subj
    )
    all_datasets.append(ds)

# Merge perfectly into a single dataset
full_dataset = ConcatDataset(all_datasets)

print(f"\n--- Full Dataset Summary ---")
print(f"Total Graphs: {len(full_dataset)}")
print(f"Num Node Features (Beta Only): {all_datasets[0].num_node_features}")
print(f"Num Edge Features: {all_datasets[0].num_edge_features}")
print(f"Classes: {list(all_datasets[0].category_to_id.keys())}")

# Inspect the very first graph of the combined dataset specifically looking at metadata
first_graph = full_dataset[0]
print("\nExample Data Object Metadata:")
print(f"Image: {first_graph.imgname}")
print(f"Subject: {first_graph.subject}, Session: {first_graph.session}, Local ID: {first_graph.local_idx}")
print(f"Target Label ID: {first_graph.y.item()}")
print(f"Shape: {first_graph}")


--- Full Dataset Summary ---
Total Graphs: 14051
Num Node Features (Beta Only): 1
Num Edge Features: 1
Classes: ['person', 'food', 'vehicle', 'location', 'object', 'clothing', 'living thing']

Example Data Object Metadata:
Image: n01930112_19568.JPEG
Subject: CSI1, Session: tensor([1]), Local ID: tensor([0])
Target Label ID: 6
Shape: Data(x=[414, 1], edge_index=[2, 16596], edge_attr=[16596], y=[1], node_id=[414], imgname='n01930112_19568.JPEG', session=[1], local_idx=[1], subject='CSI1')


## Graph Attention Network Architecture
We construct the GAT. We configure it to return three things on every forward pass:
1. **The Logits:** The final class prediction for the image.
2. **The Graph Embedding:** The flattened representation *before* the final classifier, useful for downstream similarity/clustering analysis.
3. **The Attention Weights:** The dynamic edge importances for interpretability.

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, Dropout, Embedding
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

class BOLD5000_GAT(torch.nn.Module):
    def __init__(self, num_nodes, num_classes, hidden_channels=32, heads=4, dropout=0.5):
        super(BOLD5000_GAT, self).__init__()
        
        self.dropout_rate = dropout
        
        # --- 1. Learned Node Identity ---
        # Each of the 414 regions gets a trainable 'hidden_channels' dimensional vector
        self.node_embedding = Embedding(num_nodes, hidden_channels)
        
        # --- 2. Functional Signal Embedder ---
        # Project the 1D BOLD beta into the same 'hidden_channels' dimensional space
        self.beta_embed = Linear(1, hidden_channels)
        
        # --- 3. Graph Layers ---
        self.conv1 = GATConv(
            in_channels=hidden_channels, 
            out_channels=hidden_channels, 
            heads=heads, 
            dropout=dropout,
            concat=True # Output size becomes hidden_channels * heads
        )
        
        self.conv2 = GATConv(
            in_channels=hidden_channels * heads, 
            out_channels=hidden_channels, 
            heads=heads, 
            dropout=dropout,
            concat=False # Average the heads for the final representation
        )
        
        # --- 4. Deep Classifier ---
        # We output hidden_channels * 2 because we concat Global Mean & Global Max
        self.classifier = torch.nn.Sequential(
            Linear(hidden_channels * 2, 64),
            torch.nn.ELU(),
            Dropout(p=dropout),
            Linear(64, num_classes)
        )

    def forward(self, x, edge_index, batch_index, node_id, edge_attr=None):
        """
        Forward pass.
        - x: BOLD beta signal [num_nodes * batch_size, 1]
        - edge_index: Graph connectivity matrix [2, num_edges]
        - batch_index: Tells PyG which nodes belong to which graphs in a batched tensor
        - node_id: Region identities [num_nodes * batch_size]
        """
        
        # 1. Embed functional signal
        h_beta = self.beta_embed(x)
        
        # 2. Extract structural identity
        h_struct = self.node_embedding(node_id)
        
        # 3. Fuse Identity + Function (Multiply them heavily scales identity by the activation)
        # Using addition here is also standard, but multiplication directly achieves your "beta scaling" goal safely.
        h = h_struct * h_beta
        h = F.dropout(h, p=self.dropout_rate, training=self.training)
        
        # Graph Layer 1
        h = self.conv1(h, edge_index, edge_attr=edge_attr)
        h = F.elu(h) 
        h = F.dropout(h, p=self.dropout_rate, training=self.training)
        
        # Graph Layer 2
        h, (attention_edge_index, attention_weights) = self.conv2(
            h, edge_index, edge_attr=edge_attr, return_attention_weights=True
        )
        h = F.elu(h)
        
        # 4. Smart Pooling
        # Mean pool captures the "average brain state"
        # Max pool captures the "most highly active specific regions"
        pool_mean = global_mean_pool(h, batch_index)
        pool_max = global_max_pool(h, batch_index)
        
        graph_embedding = torch.cat([pool_mean, pool_max], dim=1) # [batch_size, hidden_channels * 2]
        
        # Classification
        logits = self.classifier(graph_embedding)
        
        return logits, graph_embedding, (attention_edge_index, attention_weights)

# Instantiate the model
num_nodes = all_datasets[0][0].x.shape[0] # dynamically grab nodes size (414)
num_classes = len(all_datasets[0].category_to_id.keys())

model = BOLD5000_GAT(
    num_nodes=num_nodes,
    num_classes=num_classes,
    hidden_channels=32, 
    heads=4,
    dropout=0.5
)

print(model)

BOLD5000_GAT(
  (node_embedding): Embedding(414, 32)
  (beta_embed): Linear(in_features=1, out_features=32, bias=True)
  (conv1): GATConv(32, 32, heads=4)
  (conv2): GATConv(128, 32, heads=4)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ELU(alpha=1.0)
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=7, bias=True)
  )
)


## Data Splits & Training Loop
We split our dataset into train, validation, and test sets. Then, we set up PyTorch Geometric `DataLoader`s and define our training/evaluation logic, properly unpacking the `logits` from the model's 3-tuple return for calculating Cross-Entropy loss.

In [24]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.loader import DataLoader
from torch.utils.data import Subset
from collections import Counter
import random

# Dataset Splits
torch.manual_seed(42)
random.seed(42)

# First, separate indices by class to ensure we can balance them
class_indices = {}
for i in range(len(full_dataset)):
    label = full_dataset[i].y.item()
    if label not in class_indices:
        class_indices[label] = []
    class_indices[label].append(i)

# Find the minimum class count
min_class_count = min(len(indices) for indices in class_indices.values())
print(f"Minimum samples per class: {min_class_count}")

# Decide exact counts per class to guarantee perfect stratification
train_count = int(0.7 * min_class_count)
val_count = int(0.15 * min_class_count)
test_count = min_class_count - train_count - val_count

train_indices = []
val_indices = []
test_indices = []

# Stratified splitting: assign fixed amounts of each class to each split
for label, indices in class_indices.items():
    random.shuffle(indices)
    
    train_indices.extend(indices[:train_count])
    val_indices.extend(indices[train_count : train_count + val_count])
    test_indices.extend(indices[train_count + val_count : train_count + val_count + test_count])

# Shuffle the combined indices so batches are properly mixed
random.shuffle(train_indices)
random.shuffle(val_indices)
random.shuffle(test_indices)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)
test_dataset = Subset(full_dataset, test_indices)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2, persistent_workers=True)

print(f"Stratified Data Split| Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# Training Setup
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps') # Use Apple Silicon GPU
else:
    device = torch.device('cpu')
    
print(f"Using Device: {device}")
model = model.to(device)

# Lower learning rate to 3e-4 to allow the Embeddings to learn smoothly
optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-3)

# Handle Class Imbalance - (No longer strictly needed since we balanced the data perfectly, but kept for safety structure)
train_labels = [train_dataset[i].y.item() for i in range(len(train_dataset))]
class_counts = Counter(train_labels)
num_classes = len(all_datasets[0].category_to_id.keys())
print(f"Train Class Counts: {class_counts}")
print(f"Val Class Counts  : {Counter([val_dataset[i].y.item() for i in range(len(val_dataset))])}")

lossfunc = nn.CrossEntropyLoss()

from tqdm.auto import tqdm

# Training Logic
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    
    # Wrap loader in tqdm for a progress bar
    progress_bar = tqdm(loader, desc="Training Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        optimizer.zero_grad()
        
        # Unpack the 3-tuple, pass in data.node_id
        logits, _, _ = model(data.x, data.edge_index, data.batch, data.node_id, data.edge_attr)
        loss = criterion(logits, data.y.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs 
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Eval Logic
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    
    progress_bar = tqdm(loader, desc="Eval Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        
        logits, _, _ = model(data.x, data.edge_index, data.batch, data.node_id, data.edge_attr)
        loss = criterion(logits, data.y.view(-1))
        
        total_loss += loss.item() * data.num_graphs
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

Minimum samples per class: 381
Stratified Data Split| Train: 1862 | Val: 399 | Test: 406
Using Device: cuda
Train Class Counts: Counter({6: 266, 3: 266, 2: 266, 0: 266, 4: 266, 1: 266, 5: 266})
Val Class Counts  : Counter({3: 57, 6: 57, 2: 57, 5: 57, 0: 57, 1: 57, 4: 57})
Val Class Counts  : Counter({3: 57, 6: 57, 2: 57, 5: 57, 0: 57, 1: 57, 4: 57})


In [22]:
# Training  + Eval Loop
'''
epochs = 50
print("\nStarting Training...")
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, lossfunc)
    val_loss, val_acc = evaluate(model, val_loader, lossfunc)
    
    # Print status cleanly
    
    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    '''

'\nepochs = 50\nprint("\nStarting Training...")\nfor epoch in range(1, epochs + 1):\n    train_loss, train_acc = train_epoch(model, train_loader, optimizer, lossfunc)\n    val_loss, val_acc = evaluate(model, val_loader, lossfunc)\n\n    # Print status cleanly\n\n    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")\n'

In [27]:
## Baseline MLP (No Graphs)
#To test if the graph architecture is the bottleneck, we create a simple Multi-Layer Perceptron (MLP). It completely ignores the structural connectome and simply flattens the 414 BOLD betas into a single vector (`[batch_size, 414]`) to predict the semantic class.

class BaselineMLP(torch.nn.Module):
    def __init__(self, num_features=414, num_classes=5, dropout=0.6): # Increased dropout default to 0.6
        super(BaselineMLP, self).__init__()
        
        self.network = torch.nn.Sequential(
            Dropout(p=0.3),
            
            Linear(num_features, 128),
            torch.nn.BatchNorm1d(128),
            torch.nn.ReLU(),
            Dropout(p=dropout),
            
            Linear(128, 64),
            torch.nn.BatchNorm1d(64),
            torch.nn.ReLU(),
            Dropout(p=dropout),
            
            Linear(64, num_classes)
        )

    def forward(self, x, batch_index):
        # x is originally [nodes * batch_size, 1].
        # We need to reshape it back to [batch_size, num_nodes]
        from torch_geometric.utils import to_dense_batch
        x_dense, _ = to_dense_batch(x, batch_index)
        x_flat = x_dense.view(x_dense.size(0), -1) # Flatten to [batch_size, 414]
        
        logits = self.network(x_flat)
        return logits, None, None # Return None for embeddings/attention to match GAT signature



# Training Logic
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct = 0, 0
    
    # Wrap loader in tqdm for a progress bar
    progress_bar = tqdm(loader, desc="Training Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        optimizer.zero_grad()
        
        # MLP only needs x and batch index
        logits, _, _ = model(data.x, data.batch)
        loss = criterion(logits, data.y.view(-1))
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.num_graphs 
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Eval Logic
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct = 0, 0
    
    progress_bar = tqdm(loader, desc="Eval Batches", leave=False)
    for data in progress_bar:
        data = data.to(device)
        
        # MLP only needs x and batch index
        logits, _, _ = model(data.x, data.batch)
        loss = criterion(logits, data.y.view(-1))
        
        total_loss += loss.item() * data.num_graphs
        preds = logits.argmax(dim=1)
        correct += (preds == data.y.view(-1)).sum().item()
        
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


# Instantiate MLP 
mlp_model = BaselineMLP(num_features=414, num_classes=num_classes, dropout=0.2).to(device)

# AdamW for decoupled weight decay
mlp_optimizer = optim.AdamW(mlp_model.parameters(), lr=1e-3, weight_decay=1e-2)

# Learning Rate Scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(mlp_optimizer, mode='min', factor=0.5, patience=5)

print(mlp_model)

# Train MLP Baseline
epochs_mlp = 60
print("\nStarting MLP Baseline Training with Strong Regularization...")
for epoch in range(1, epochs_mlp + 1):
    train_loss, train_acc = train_epoch(mlp_model, train_loader, mlp_optimizer, lossfunc)
    val_loss, val_acc = evaluate(mlp_model, val_loader, lossfunc)
    
    # Step the scheduler based on validation loss
    scheduler.step(val_loss)
    current_lr = mlp_optimizer.param_groups[0]['lr']
    
    print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} || Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

BaselineMLP(
  (network): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=414, out_features=128, bias=True)
    (2): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU()
    (4): Dropout(p=0.2, inplace=False)
    (5): Linear(in_features=128, out_features=64, bias=True)
    (6): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
    (9): Linear(in_features=64, out_features=7, bias=True)
  )
)

Starting MLP Baseline Training with Strong Regularization...


Epoch 001 | Train Loss: 1.9883 | Train Acc: 0.1676 || Val Loss: 1.9419 | Val Acc: 0.1554 | LR: 0.001000


Epoch 002 | Train Loss: 1.8641 | Train Acc: 0.2476 || Val Loss: 1.9010 | Val Acc: 0.2155 | LR: 0.001000


Epoch 003 | Train Loss: 1.8009 | Train Acc: 0.2900 || Val Loss: 1.8665 | Val Acc: 0.2406 | LR: 0.001000


Epoch 004 | Train Loss: 1.7287 | Train Acc: 0.3324 || Val Loss: 1.8390 | Val Acc: 0.2406 | LR: 0.001000


...

Epoch 056 | Train Loss: 0.9926 | Train Acc: 0.6622 || Val Loss: 1.9754 | Val Acc: 0.2807 | LR: 0.000004


Epoch 057 | Train Loss: 0.9758 | Train Acc: 0.6654 || Val Loss: 1.9643 | Val Acc: 0.2732 | LR: 0.000004


Epoch 058 | Train Loss: 1.0133 | Train Acc: 0.6321 || Val Loss: 1.9583 | Val Acc: 0.2682 | LR: 0.000004


Epoch 059 | Train Loss: 1.0175 | Train Acc: 0.6332 || Val Loss: 1.9622 | Val Acc: 0.2807 | LR: 0.000004


Epoch 060 | Train Loss: 1.0127 | Train Acc: 0.6375 || Val Loss: 1.9734 | Val Acc: 0.2707 | LR: 0.000004
